# Gerber Beam with Internal Hinges

This tutorial demonstrates a planar Gerber beam using the LowLevelFEM weak-form DSL.

The structure is described with a two-field Timoshenko beam formulation:

- the displacement field $\mathbf{u}=[u_x,u_y]^T$,
- the independent cross-section rotation $\varphi$.

The internal hinges are modeled with multi-point constraints (MPCs). Translational continuity is enforced at the hinge, while the rotations of the connected beam segments remain independent. This reproduces the kinematics of an ideal internal hinge.


In [3]:
using LowLevelFEM

## Geometry

The geometry and mesh are read from `Gerber.geo`. The beam is divided into three physical curve groups: `ABC`, `CDE`, and `EFG`.

At each internal hinge, two coincident but topologically independent nodes are used. This makes it possible to couple translations while leaving the rotational degrees of freedom uncoupled.


In [4]:
openGeometry("Gerber.geo")
#openPreProcessor()

The hinge coordinates are stored as Gmsh parameters and can be recovered directly from the model. They can be useful later for probing or plotting results at the hinge locations.


In [5]:
xC = getParameter("xC")
xE = getParameter("xE");

## Fields and materials

Each beam segment is associated with a `Material` object. Two unknown fields are introduced:

- `U`: the two-component translational displacement field,
- `Φ`: the scalar rotation field.

The rotation field is created with `reducedOrder=true`, so it is solved in a one-order-lower continuous Lagrange space while assembly and storage remain associated with the original mesh.


In [6]:
mat1 = Material("ABC")
mat2 = Material("CDE")
mat3 = Material("EFG")
U = Field([mat1,mat2,mat3], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
Φ = Field([mat1,mat2,mat3], type=:ScalarField, dim=2, fieldName=:φ, rhsName=:m, reducedOrder=true);

## Local beam directions

The beam equations are written in a local tangent-normal frame. The unit tangent vector is

$$
\mathbf{t}=
\begin{bmatrix}
t_x\\
t_y
\end{bmatrix},
$$

and the in-plane unit normal is

$$
\mathbf{n}=
\begin{bmatrix}
-t_y\\
t_x
\end{bmatrix}.
$$

Because the tangent is evaluated elementwise, the same formulation can also be used on curved beam segments.


In [7]:
t = tangentVector(U, "ABC") + tangentVector(U, "CDE") + tangentVector(U, "EFG")
tx, ty, tz = t[1], t[2], t[3]

nx = -ty
ny = tx
nz = 0;

## Timoshenko beam kinematics

The generalized beam strain vector is

$$
\boldsymbol{\varepsilon}_b=
\begin{bmatrix}
\varepsilon\\
\gamma\\
\kappa_b
\end{bmatrix},
$$

with axial strain

$$
\varepsilon=\mathbf{t}^T(\nabla\mathbf{u})\mathbf{t},
$$

shear strain

$$
\gamma=\mathbf{n}^T(\nabla\mathbf{u})\mathbf{t}-\varphi,
$$

and curvature

$$
\kappa_b=\nabla\varphi\cdot\mathbf{t}.
$$

The matrices `Au`, `Aφ`, and `Gφ` map the LowLevelFEM operator output to these generalized strain components.


In [8]:
# ------------------------------------------------------------------
# Generalized Timoshenko beam strain
#
# ε = tᵀ ∇u t
# γ = nᵀ ∇u t - φ
# κ = ∇φ ⋅ t
# ------------------------------------------------------------------

Au = [
    tx*tx tx*ty tx*ty ty*ty
    nx*tx nx*ty ny*tx ny*ty
    0 0 0 0]

Aφ = [
    0 0
    0 0
    tx ty]

Gφ = [0; -1; 0;;]

Bu = Au ⋅ Grad(U)

Bφ = Aφ ⋅ Grad(Φ) + Gφ ⋅ Φ;

## Cross-section properties

For the rectangular cross-section,

$$
A=bh,
\qquad
I_z=\frac{bh^3}{12}.
$$

The Timoshenko shear correction factor is chosen as $\kappa_s=4/5$.


In [9]:
b = 30
h = 5
E = mat1.E
G = mat1.μ
Iz = b * h^3 / 12
κ = 4 / 5
A = b * h;

The constitutive relation between generalized strains and section resultants is

$$
\begin{bmatrix}
N\\
T\\
M
\end{bmatrix}
=
D
\begin{bmatrix}
\varepsilon\\
\gamma\\
\kappa_b
\end{bmatrix},
$$

where

$$
D=
\begin{bmatrix}
EA & 0 & 0\\
0 & \kappa_sGA & 0\\
0 & 0 & EI_z
\end{bmatrix}.
$$


In [10]:
D = [
    E*A 0 0
    0 κ*G*A 0
    0 0 E*Iz]

3×3 Matrix{Float64}:
 3.0e7  0.0        0.0
 0.0    9.23077e6  0.0
 0.0    0.0        6.25e7

## Weak form and stiffness matrix

The internal virtual work is

$$
\delta W_{\mathrm{int}}
=
\int_\Gamma
\delta\boldsymbol{\varepsilon}_b^T
D
\boldsymbol{\varepsilon}_b\,\mathrm{d}s.
$$

With the combined operator $B=B_u+B_\varphi$, the stiffness matrix follows from

$$
K=\int_\Gamma B^TDB\,\mathrm{d}s.
$$

The three beam segments are integrated separately and then added. Since `B` contains operators associated with both fields, LowLevelFEM automatically assembles the corresponding multifield block matrix.


In [11]:
B = Bu + Bφ
K = ∫(B' ⋅ D ⋅ B, Γ="ABC") + ∫(B' ⋅ D ⋅ B, Γ="CDE") + ∫(B' ⋅ D ⋅ B, Γ="EFG");

## Supports and internal hinges

The external supports are imposed with standard `BoundaryCondition` objects.

At the internal hinges, translational continuity is enforced with

$$
\mathbf{u}_{C_1}=\mathbf{u}_{C_2},
\qquad
\mathbf{u}_{E_1}=\mathbf{u}_{E_2}.
$$

Only the displacement field is constrained by the MPCs. The rotation field $\varphi$ remains independent on the two sides of each hinge, so the hinge does not transfer bending moment.


In [12]:
supp_A = BoundaryCondition("A", field=U, ux=0, uy=0)
supp_B = BoundaryCondition("B", field=U, uy=0)
supp_D = BoundaryCondition("D", field=U, uy=0)
supp_G = BoundaryCondition("G", field=U, uy=0)

mpc_C = MPC(master="C1", slave="C2", field=U)
mpc_E = MPC(master="E1", slave="E2", field=U)

MultiPointConstraint("E1", "E2", Problem("Gerber", :VectorField, 2, 2, Material[Material("ABC", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("CDE", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("EFG", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 183, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false), Dict{Symbol, Bool}())

## External loads

The structure is loaded by

- a distributed vertical load on `AB`,
- a concentrated vertical force at `E1`,
- a concentrated moment at `F`.

The first implementation uses the classical `LoadCondition` and `loadVector` interface. The displacement- and rotation-field load vectors are collected into a single `SystemVector`.


In [13]:
loadf = LoadCondition("AB", fy=-1)
loadF = LoadCondition("E1", fy=-100)
loadM = LoadCondition("F", m=10000)

fu1 = loadVector(U, [loadf, loadF])
fφ1 = loadVector(Φ, [loadM])

F = SystemVector([fu1, fφ1]);

The same loading can also be written directly with the weak-form integration syntax. This form mirrors the mathematical loading terms closely.

For a concentrated moment $M_F$ acting on the rotation field,

$$
\delta W_M=\delta\varphi_F\,M_F.
$$

`F1` is therefore an equivalent DSL-based representation of the same external loading.


In [14]:
fu = ∫(U ⋅ [0,-1], Γ="AB") + ∫(U ⋅ [0,-100], Γ="E1")
fφ = ∫(Φ ⋅ 10000, Γ="F")

F1 = SystemVector([fu, fφ]);

## Solution

The coupled multifield system is solved with `solveField`. The support conditions and the two hinge MPCs are passed independently.

The result consists of the displacement field $\mathbf{u}$ and the independent rotation field $\varphi$.


In [15]:
u, φ = solveField(K, F, support=[supp_A, supp_B, supp_D, supp_G], mpc=[mpc_C, mpc_E]);

The translational displacement field can be visualized directly in Gmsh. A scale factor is used here to make the deformation easier to inspect.


In [16]:
showDoFResults(fu, name="force")
showDoFResults(u, name="u", factor=20, visible=true);

## Recovery of beam resultants

For post-processing, the local in-plane normal vector is stored as a `VectorField`. The generalized beam strains are then recovered from the solved displacement and rotation fields.


In [17]:
n = VectorField([nx, ny, 0]);

The displacement field is expanded to the 3D vector representation used by the general gradient and vector-algebra routines. The rotation field is converted to an elementwise field before its gradient is evaluated.

The recovered quantities are

$$
\varepsilon=\mathbf{t}\cdot\left((\nabla\mathbf{u})\mathbf{t}\right),
$$

$$
\gamma=\mathbf{n}\cdot\left((\nabla\mathbf{u})\mathbf{t}\right)-\varphi,
$$

and

$$
\kappa_b=\nabla\varphi\cdot\mathbf{t}.
$$

The corresponding internal forces are

$$
N=EA\varepsilon,
\qquad
T=\kappa_sGA\gamma,
\qquad
M=EI_z\kappa_b.
$$


In [18]:
u = expandTo3D(u)
φ = nodesToElements(φ)

g = ∇(u)

ε = t ⋅ (g * t)
γ = n ⋅ (g * t) - φ

κb = ∇(φ) ⋅ t

N = E * A * ε
T = κ * G * A * γ
Mh = E * Iz * κb;

## Internal-force diagrams

`plotOnBeam` restricts each elementwise result to the selected physical beam segment and creates an offset diagram along the beam axis.

Axial force $N$, shear force $T$, and bending moment $M$ are plotted separately for the three beam parts.


In [19]:
N0, plotN1 = plotOnBeam("ABC", N, name="N1 graph", plot=true)
T0, plotT1 = plotOnBeam("ABC", T, name="T1 graph", plot=true)
Mh0, plotMh1 = plotOnBeam("ABC", Mh, name="Mh1 graph", plot=true)

N1, plotN2 = plotOnBeam("CDE", N, name="N2 graph", plot=true)
T1, plotT2 = plotOnBeam("CDE", T, name="T2 graph", plot=true)
Mh1, plotMh2 = plotOnBeam("CDE", Mh, name="Mh2 graph", plot=true)

N2, plotN3 = plotOnBeam("EFG", N, name="N3 graph", plot=true)
T2, plotT3 = plotOnBeam("EFG", T, name="T3 graph", plot=true)
Mh2, plotMh3 = plotOnBeam("EFG", Mh, name="Mh3 graph", plot=true);

The returned diagram coordinates can also be processed with an external plotting package if a publication-style figure is preferred.


In [20]:
using Plots

LoadError: ArgumentError: Package Plots not found in current path.
- Run `import Pkg; Pkg.add("Plots")` to install the Plots package.

In [21]:
plot(plotN1)
plot!(plotN2[1].+xC, plotN2[2])
plot!(plotN3[1].+xE, plotN3[2])

LoadError: UndefVarError: `plot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [22]:
plot(plotT1)
plot!(plotT2[1].+xC, plotT2[2])
plot!(plotT3[1].+xE, plotT3[2])

LoadError: UndefVarError: `plot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [23]:
plot(plotMh1)
plot!(plotMh2[1].+xC, plotMh2[2])
plot!(plotMh3[1].+xE, plotMh3[2])

LoadError: UndefVarError: `plot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [24]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
